### Imports

In [1]:
import chromadb
import random
import time
import pandas as pd
from pathlib import Path
import re
import json
import subprocess
from collections import Counter
from sentence_transformers import SentenceTransformer
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_ollama import ChatOllama, OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.language_models.chat_models import BaseChatModel

### Setup

In [22]:
PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLk1fjOl39-50kWobutO8NVFzbw9PHtbbg"

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
api = YouTubeTranscriptApi()

client = chromadb.PersistentClient(path="./chroma_easy_german")
col = client.get_or_create_collection(
    "easy_german_a1_multi",
    metadata={"hnsw:space": "cosine"}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Helper Functions

In [23]:
def is_subtitles_disabled(e: Exception) -> bool:
    return "subtitles are disabled" in str(e).lower()

def is_no_transcript(e: Exception) -> bool:
    return "no transcripts were found" in str(e).lower()

In [24]:
def is_ip_block_error(e: Exception) -> bool:
    msg = str(e).lower()
    return (
        "blocking requests from your ip" in msg
        or "requestblocked" in msg
        or "ipblocked" in msg
        or "too many requests" in msg
        or "toomanyrequests" in msg
    )

In [25]:
def fetch_with_backoff(video_id: str, languages=("de",), max_retries=6):
    delay = 20.0
    for attempt in range(max_retries):
        try:
            return api.fetch(video_id, languages=list(languages))
        except Exception as e:
            # If it's NOT an IP-block style error, bubble up (subtitles disabled, no transcript, etc.)
            if not is_ip_block_error(e):
                raise

            sleep_s = delay + random.uniform(0, 1.5)
            print(f"[BLOCKED] {video_id} — sleeping {sleep_s:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(sleep_s)
            delay *= 2

    raise RuntimeError(f"Still blocked after {max_retries} retries for {video_id}")

In [26]:
def get_playlist_entries(playlist_url: str):
    """
    Returns a list of dicts: 
    [{"id": "...", "title": "...", "url": "..."}]
    Uses yt-dlp.
    """
    cmd = [
        "yt-dlp",
        "--flat-playlist",
        "--dump-single-json",
        playlist_url,
    ]
    out = subprocess.check_output(cmd, text=True)
    data = json.loads(out)

    entries = []
    for e in data.get("entries", []):
        vid = e.get("id")
        if not vid:
            continue
        entries.append({
            "id": vid,
            "title": e.get("title", ""),
            "url": f"https://www.youtube.com/watch?v={vid}",
        })
    return entries

In [27]:
def get_playlist_entries_from_csv(path: str):
    df = pd.read_csv(path)

    entries = []
    for _, row in df.iterrows():
        entries.append({
            "id": row["id"],
            "title": row.get("title", ""),
            "url": row.get("url", f"https://www.youtube.com/watch?v={row['id']}")
        })

    return entries

In [48]:
def merge_transcript_chunks(
    chunks,
    max_chars=900,
    min_chars=300
    ):
    """
    Merge micro transcript segments into RAG-sized
    chunks while preserving timestamps.
    """
    merged = []
    buf_text = []
    buf_start = None
    buf_end = None
    
    for chunk in chunks:
        t = chunk["text"].strip()
        if not t:
            continue
        if buf_start is None:
            buf_start = chunk["start"]
            
        candidate = (" ".join(buf_text + [t])).strip()
    
        if len(candidate) < max_chars:
            buf_text.append(t)
            buf_end = chunk["end"]
        else:
            if buf_text:
                merged.append({
                    "text": " ".join(buf_text).strip(),
                    "start": buf_start,
                    "end": buf_end
                            })
            buf_text = [t]
            buf_start = chunk["start"]
            buf_end = chunk["end"]
            
    if buf_text:
        merged.append({
            "text": " ".join(buf_text).strip(),
            "start": buf_start,
            "end": buf_end})
    
    merged = [m for m in merged if len(m["text"]) >= min_chars]
    return merged

### Get entries to parse the videos

In [ ]:
# entries = get_playlist_entries(PLAYLIST_URL)
entries = get_playlist_entries_from_csv("entries.csv")

In [30]:
print(entries)

[{'id': 'huwi-cjPPXU', 'title': 'Introduce Yourself in Slow German | Super Easy German 258', 'url': 'https://www.youtube.com/watch?v=huwi-cjPPXU'}, {'id': 'Yaelm87PTvg', 'title': 'Introduce yourself in German (for absolute beginners) | Super Easy German (76)', 'url': 'https://www.youtube.com/watch?v=Yaelm87PTvg'}, {'id': 'hAkxKMlYUI4', 'title': 'German Alphabet & Pronunciation - Full Guide | Super Easy German 253', 'url': 'https://www.youtube.com/watch?v=hAkxKMlYUI4'}, {'id': 'LQiHX6OY_BI', 'title': 'Our Morning Routine in Slow German | Super Easy German 232', 'url': 'https://www.youtube.com/watch?v=LQiHX6OY_BI'}, {'id': 'aRlakaPVrEw', 'title': 'Learn Basic German Greetings & Farewells in Slow German | Super Easy German 274', 'url': 'https://www.youtube.com/watch?v=aRlakaPVrEw'}, {'id': '9h8p08qziG0', 'title': 'Conjugation of regular verbs: Sagen, Machen, Hören | Super Easy German (83)', 'url': 'https://www.youtube.com/watch?v=9h8p08qziG0'}, {'id': 'uO0jWxhVW1A', 'title': 'Easy German 

In [31]:
df = pd.read_csv("entries.csv")
id2title = dict(zip(df["id"], df["title"]))

### Loop through videos

In [32]:
# Get all existing video_ids in collection
existing = col.get(include=["metadatas"])

existing_video_ids = set()

for meta in existing["metadatas"]:
    if meta and "video_id" in meta:
        existing_video_ids.add(meta["video_id"])

print("Already indexed videos:", len(existing_video_ids))

Already indexed videos: 80


In [33]:
for idx, e in enumerate(entries, start=1):
    video_id = e["id"]
    title = e["title"]

    if video_id in existing_video_ids:
        print(f"[{idx}/{len(entries)}] SKIP {video_id} — already indexed")
        continue

    try:
        transcript_de = fetch_with_backoff(video_id, languages=("de",))
    except Exception as e:
        print(f"SKIP {video_id}: {e}")
        continue
    # polite pacing between successful requests too
    time.sleep(random.uniform(1.5, 10.0))
        
    structured_chunks = [
    {"text": chunk.text,
     "start": chunk.start,
     "end": chunk.start + chunk.duration
     } 
    for chunk in transcript_de
    ]
    
    merged_chunks = merge_transcript_chunks(chunks=structured_chunks, max_chars=600)
    if not merged_chunks:
        print(f"[{idx}/{len(entries)}] SKIP {video_id} — merged_chunks empty")
        continue
    
    docs = [c["text"] for c in merged_chunks]
    metas = [
    {
        "video_id": video_id,
        "title": title,
        "lang": "de",
        "start": c["start"],
        "end": c["end"],
        "playlist": "easy_german_a1",
    } 
    for c in merged_chunks
    ]
    
    embs = model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
    ).tolist()

    ids = [f"{video_id}_{i}" for i in range(len(docs))]

    if hasattr(col, "upsert"):
        col.upsert(
            documents=docs,
            embeddings=embs,
            ids=ids,
            metadatas=metas
            )
    else:
        col.add(
            documents=docs,
            embeddings=embs,
            ids=ids,
            metadatas=metas
            )
        
    print(f"[{idx}/{len(entries)}] OK {video_id} — {len(docs)} chunks — {title}")
        

[1/83] SKIP huwi-cjPPXU — already indexed
[2/83] SKIP Yaelm87PTvg — already indexed
[3/83] SKIP hAkxKMlYUI4 — already indexed
[4/83] SKIP LQiHX6OY_BI — already indexed
[5/83] SKIP aRlakaPVrEw — already indexed
[6/83] SKIP 9h8p08qziG0 — already indexed
[7/83] SKIP uO0jWxhVW1A — already indexed
[8/83] SKIP QNq1Xp6DgJw — already indexed
[9/83] SKIP Q3eCDhwMRv8 — already indexed
[10/83] SKIP tNrwiUGHMiU — already indexed
[11/83] SKIP MqqHc6lRQG4 — already indexed
[12/83] SKIP r94aqLUO0wo — already indexed
[13/83] SKIP QCGdeH4hYdo — already indexed
[14/83] SKIP NsqA8_SmdVI — already indexed
[15/83] SKIP o-Zu-bUlPb4 — already indexed
SKIP ITjyfCAspco: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=ITjyfCAspco! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-tr

In [ ]:
def return_context(query: str, k: int=3):
    '''
    A function to return the context of the k closest based on cosine similarity
    '''
    q_emb = model.encode([query], normalize_embeddings=True).tolist()

    res = col.query(query_embeddings=q_emb, n_results=5)

    ctx = []

    for meta, doc in zip(res["metadatas"][0], res["documents"][0]):
        vid = meta.get("video_id")
        title = meta.get("title") or id2title.get(vid) or vid or "Unknown"
        ctx.append(f"{title} [{meta.get('start',0):.1f}s–{meta.get('end',0):.1f}s]{doc}")
    return "\n\n".join(ctx)

### Creating the LLM (coach layer)

In [61]:
query = "How do Germans apologize in daily life?"

In [ ]:
llm_coach_chat = ChatOllama(
    model="llama3.1:latest", 
    temperature=0, 
    num_ctx=1024,
    num_predict=120,
    )

llm_fast = ChatOllama(
    model="qwen2.5:3b",
    temperature=0,
    num_ctx=1024,
    num_predict=120
)

### Tools for my agent to call

In [63]:
def tutor_a1(llm, user_query: str, k: int = 2) -> str:
    t0 = time.time()
    ctx = return_context(user_query, k=k)
    t1 = time.time()
    ctx = return_context(user_query, k=k)
    ctx = ctx[:1200]

    system = SystemMessage(content="""
You are a structured German tutor (A1).
Return EXACTLY:
- 3 phrases about the query (DE—EN)
- 2 examples (DE—EN)
- 1 short follow up
Maximum 80 words total.
Do not add explanations.
""".strip())

    user = HumanMessage(content=f"""
Question: {user_query}

Context:
{ctx}
""".strip())

    out = llm.invoke([system, user])
    t2 = time.time()
    print(f"retrieval: {t1-t0:.2f}s | llm: {t2-t1:.2f}s | ctx_chars={len(ctx)}")
    return out.content if hasattr(out, "content") else out

print(tutor_a1(llm_fast, query, k=2))

retrieval: 0.14s | llm: 11.98s | ctx_chars=1200
Phrases:
1. Germans often say "Es tut mir leid" when they apologize.
2. They might add "Vielen Dank für eure Verständnis."
3. Sometimes, they use "Ich schätze es sehr."

Examples:
1. Ich habe vergessen zu sagen, vielen Dank für euer Verständnis.
2. Es tut mir leid, dass ich spät gekommen bin.

Follow up: 
Haben Sie noch Fragen zum Thema Deutsch oder Apologetik?


In [64]:
t0 = time.time()
ans = tutor_a1(llm_fast, query)
t1 = time.time()

print("seconds:", t1 - t0)
print("tokens approx:", len(ans.split()))
print("approx tok/sec:", len(ans.split()) / (t1 - t0))

retrieval: 0.20s | llm: 15.47s | ctx_chars=1200
seconds: 15.665279150009155
tokens approx: 70
approx tok/sec: 4.468480856912089


retrieval: 0.14s | llm: 22.28s | ctx_chars=1200
seconds: 22.42194128036499
tokens approx: 60
approx tok/sec: 2.6759502778888424

### Building german agent

In [ ]:
def german_agent(llm, user_query: str, mode: str='tutor', level: str='a1', k: int=2):
    """
    Main orchestration function for the German learning agent.

    This function:
    - Selects the appropriate instructional mode.
    - Adjusts output complexity based on CEFR level.
    - Processes the user query.
    - Retrieves and incorporates relevant transcript context.

    Parameters
    ----------
    mode : {'tutor', 'vocab', 'grammar', 'quiz'}
        Determines the type of learning interaction:
        - 'tutor'   : Structured explanation and guided practice
        - 'vocab'   : Vocabulary extraction with examples
        - 'grammar' : Grammar explanation and pattern breakdown
        - 'quiz'    : Assessment-style exercises

    level : {'A1', 'A2', 'B1', 'B2', 'C1', 'C2'}
        CEFR proficiency level used to control linguistic complexity.

    user_query : str
        The user’s question or topic of interest.

    Returns
    -------
    str
        A structured response generated by the language model
        according to the selected mode and level.
    """
    
    templates = {
    "tutor": f"""
        You are a supportive German language tutor teaching at {level} level.

        Follow these steps:

        1. Answer the student's question clearly.
        2. Explain why the answer is correct.
        3. Use relevant examples from the provided context.
        4. Give 2–3 short German examples with English translations.
        5. End with one guiding follow-up question to encourage thinking.

        Use simple language appropriate for {level}.
        Respond only in English and German.
        Do not mention sources or timestamps.
            """,

     "vocab": f"""
        You are a structured German vocabulary trainer ({level} level).
        Return EXACTLY:
        - 5 most key related to the query (DE-EN)
        - 1 example sentence for each from the context (DE-EN)
        Maximum 80 words total.
        Do not add explanations.
            """,
        
    "grammar": f"""
        You are a structured German grammar trainer ({level} level).
        Return EXACTLY:
        - 3 example sentences from the context (DE-EN)
        - A short grammar explanation (DE-EN)
        Maximum 80 words total.
            """,
        
    "quiz": f"""
        You are a structured German quiz generator ({level} level).
        Return EXACTLY:
        - 3 short quiz questions from the context (DE—EN)
        Maximum 80 words total.
        Do not add explanations.
            """
    }
                                
    if mode not in templates:
        raise ValueError(f"Invalid mode: {mode}")    

    ctx = return_context(query, k=k)
    
    system_text = templates[mode].strip()
        
    system = SystemMessage(content = system_text)
            
    user = HumanMessage(content=f"""
                                Question: {user_query}

                                Context:
                                {ctx}
                                """.strip())

    out = llm.invoke([system, user])
    return out.content if hasattr(out, "content") else out

In [80]:
query = "What is the difference of accusative and nominative?"

print(german_agent(llm_fast, query, 'tutor', 'A1'))

print("\n-------------------------------------\n")

print(german_agent(llm_coach_chat, query, 'tutor', 'A1'))

- Akkusativ — Nominativ — Der Akkusativ und der Nominativ sind zwei Fälle im Deutschen, die unterschiedliche Verwendung haben.
- Akkusativ — Accusative — In der Sprache wird der Akkusativ für den Zweck verwendet, während der Nominativ oft als Objekt oder Ziel fungiert.
- German sentence — English translation
Akkusativ und Nominativ sind zwei Fälle im Deutschen, die unterschiedliche Verwendung haben. Explanation: Akkusativ wird für den Zweck verwendet, während der Nominativ oft als Objekt oder

-------------------------------------

- Nominativ (Nominative) — Subjekt (Subject)
- Akkusativ (Accusative) — Objekt (Object)

Explanation:
In German, the nominative case is used for the subject of a sentence, while the accusative case is used for the direct object. For example: "Der Hund biss den Mann" (The dog bit the man). Here, "der Hund" is in the nominative case because it's the subject, and "den Mann" is in the accusative case because it's the direct object.

Follow-up quiz:
What is


#### Some tests

In [81]:
llm_fast = ChatOllama(
    model="qwen2.5:3b",
    temperature=0,
    num_ctx=1024,
    num_predict=120
)

def clean_context(ctx):
    lines = ctx.split("\n")
    filtered = [
        line for line in lines
        if not ("|" in line and "[" in line)  # remove metadata lines
    ]
    return "\n".join(filtered)

def german_agent(llm, user_query: str, mode: str='tutor', level: str='a1', k: int=2):
    """
    Main orchestration function for the German learning agent.

    This function:
    - Selects the appropriate instructional mode.
    - Adjusts output complexity based on CEFR level.
    - Processes the user query.
    - Retrieves and incorporates relevant transcript context.

    Parameters
    ----------
    mode : {'tutor', 'vocab', 'grammar', 'quiz'}
        Determines the type of learning interaction:
        - 'tutor'   : Structured explanation and guided practice
        - 'vocab'   : Vocabulary extraction with examples
        - 'grammar' : Grammar explanation and pattern breakdown
        - 'quiz'    : Assessment-style exercises

    level : {'A1', 'A2', 'B1', 'B2', 'C1', 'C2'}
        CEFR proficiency level used to control linguistic complexity.

    user_query : str
        The user’s question or topic of interest.

    Returns
    -------
    str
        A structured response generated by the language model
        according to the selected mode and level.
    """
    
    templates = {
    "tutor": f"""
        You are a supportive German language tutor teaching at {level} level.

        Follow these steps:

        1. Answer the student's question clearly.
        2. Explain why the answer is correct.
        3. Use relevant examples from the provided context.
        4. Give 2–3 short German examples with English translations.
        5. End with one guiding follow-up question to encourage thinking.

        Use simple language appropriate for {level}.
        Respond only in English and German.
        Do not mention sources or timestamps.
            """,

     "vocab": f"""
        You MUST respond only in English and German.
        You are a structured German vocabulary trainer ({level} level).
        Return EXACTLY:
        - 5 most key related to the query (DE-EN)
        - 1 example sentence for each from the context (DE-EN)
        Maximum 80 words total.
        Do not add explanations.
            """,
        
    "grammar": f"""
        You MUST respond only in English and German.
        You are a structured German grammar trainer ({level} level).
        Return EXACTLY:
        - 3 example sentences from the context (DE-EN)
        - A short grammar explanation (DE-EN)
        Maximum 80 words total.
            """,
        
    "quiz": f"""
        You MUST respond only in English and German.
        You are a structured German quiz generator ({level} level).
        Return EXACTLY:
        - 3 short quiz questions from the context (DE—EN)
        Maximum 80 words total.
        Do not add explanations.
            """
    }
                                
    if mode not in templates:
        raise ValueError(f"Invalid mode: {mode}")    

    ctx = clean_context(return_context(user_query, k=k))
    ctx = ctx[:800]
    
    system_text = templates[mode].strip()
        
    system = SystemMessage(content = system_text)
            
    user = HumanMessage(content=f"""
                                Question: {user_query}

                                Context:
                                {ctx}
                                """.strip())

    out = llm.invoke([system, user])
    return out.content if hasattr(out, "content") else out


query = "What is the difference of accusative and nominative?"

t0 = time.time()
print(german_agent(llm_fast, query, 'tutor', 'A1'))
t1 = time.time()
print("seconds:", t1 - t0)
print("\n-------------------------------------\n")
print(german_agent(llm_coach_chat, query, 'tutor', 'A1'))
t2 = time.time()
print("seconds:", t2 - t1)

In German, "nominative" and "accusative" are cases used with nouns.

Nominative case:
- It's used when a noun is the subject of a sentence.
- Example: Ich esse Brot. (I eat bread.) Here, "Brot" is in nominative case because it's the main person or thing doing something.

Accusative case:
- It's used when a noun is the direct object of a verb.
- Example: Ich esse das Brot. (I eat the bread.) Here, "das Brot" is in accusative
seconds: 25.652050018310547

-------------------------------------

Don't worry, it's easy!

The main difference between accusative and nominative is how we use them to talk about people or things in a sentence.

**Nominative** is used for the subject of the sentence, which means the person or thing that does something. For example:

* Ich esse ein Sandwich. (I eat a sandwich.)
In this sentence, "Ich" (I) is the subject, so it's in nominative form.

**Accusative** is used for the object of the sentence, which means the person or thing that receives an action. For ex

### Creating the topics for other modes

I could have added the topics from the video, but they are a little messy

In [7]:

PATH = ("data/entries.csv")

# I made this simple, it's just titles
# We can make it more complex later
STOPWORDS = {"easy", "german", "learn", 
             "easygerman", "episode", 
             "video", "with", "in", "the",
             "a", "an", "and", "for", 
             "how", "to", "about"}

# Had to add this, or else the topics could be strange
TOPIC_RULES = {
    "Alphabet": ["alphabet", "letter", "letters", "pronunciation"],
    "Numbers": ["counting", "number", "numbers", "counter", "numeral", "numerals"],
    "Introductions": ["introduce", "introduction", "greetings", "greeting", "hello", "hi", "good morning", "pleasure"],
    "Verbs": ["verbs", "sein", "gehen", "fahren", "finden", "geben", "wissen", "machen", "haben", "können", "wollen", "mögen"],
    "Cases": ["case", "cases", "nominativ", "akkusativ", "accusative", "dative"],
    "Prepositions": ["preposition", "präpositionen", "mit", "bei", "ohne"],
    "Restaurant & Food": ["restaurant", "order", "coffee", "drinks", "breakfast", "lunch", "dinner"],
    "Daily Life": ["daily", "morning", "vacation", "winter", "berlin"],
    "Shopping": ["supermarket", "shopping", "grocery", "mall"],
    "Conversation": ["conversation", "small talk", "phrases", "greetings"],
    "Grammar": ["conjugation", "plural", "tense", "imperative"],
    "Vocabulary": ["vocabulary", "fruits", "clothing", "emotions"]
}

In [8]:
df = pd.read_csv(PATH)
df.head()

,id,title,url
0,huwi-cjPPXU,Introduce Yourself in Slow German | Super Easy...,https://www.youtube.com/watch?v=huwi-cjPPXU
1,Yaelm87PTvg,Introduce yourself in German (for absolute beg...,https://www.youtube.com/watch?v=Yaelm87PTvg
2,hAkxKMlYUI4,German Alphabet & Pronunciation - Full Guide |...,https://www.youtube.com/watch?v=hAkxKMlYUI4
3,LQiHX6OY_BI,Our Morning Routine in Slow German | Super Eas...,https://www.youtube.com/watch?v=LQiHX6OY_BI
4,aRlakaPVrEw,Learn Basic German Greetings & Farewells in Sl...,https://www.youtube.com/watch?v=aRlakaPVrEw


In [9]:
def assign_topic(title: str):
    """Function to assign a topic due to a video title

    Args:
        title (str): The video title from the dataset

    Returns:
        _type_: It return the topic of the title
    """
    title_lower = title.lower()

    for topic, keywords in TOPIC_RULES.items():
        if any(k in title_lower for k in keywords):
            return topic

    return "Other"

In [10]:
df["topic"] = df["title"].apply(assign_topic)
df.head()

,id,title,url,topic
0,huwi-cjPPXU,Introduce Yourself in Slow German | Super Easy...,https://www.youtube.com/watch?v=huwi-cjPPXU,Introductions
1,Yaelm87PTvg,Introduce yourself in German (for absolute beg...,https://www.youtube.com/watch?v=Yaelm87PTvg,Introductions
2,hAkxKMlYUI4,German Alphabet & Pronunciation - Full Guide |...,https://www.youtube.com/watch?v=hAkxKMlYUI4,Alphabet
3,LQiHX6OY_BI,Our Morning Routine in Slow German | Super Eas...,https://www.youtube.com/watch?v=LQiHX6OY_BI,Daily Life
4,aRlakaPVrEw,Learn Basic German Greetings & Farewells in Sl...,https://www.youtube.com/watch?v=aRlakaPVrEw,Introductions


In [13]:
df.to_csv(PATH, index=False)